In [2]:

# 1. Import libraries
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from sklearn.metrics.pairwise import cosine_similarity


# 2. Load Twitter dataset
data = pd.read_csv(r'D:\Downloads\LP IV Updated Datasets\LP IV Updated Datasets\CBOW\twitter_dataset.csv')

print("Dataset Shape:", data.shape)

tweets = data['Text'].dropna().astype(str).reset_index(drop=True)


# 3. Preprocessing function
def preprocess(text):

    text = text.lower()

    text = re.sub(r'http\S+|www\S+', '', text)

    text = re.sub(r'@\w+', '', text)

    text = re.sub(r'#', '', text)

    tokens = simple_preprocess(text, deacc=True)

    return tokens


# 4. Tokenize all tweets
tokenized_tweets = [preprocess(tweet) for tweet in tweets]

tokenized_tweets = [t for t in tokenized_tweets if len(t) > 0]

print("\nSample Tokenized Tweets:")
print(tokenized_tweets[:3])


# 5. Train CBOW Word2Vec Model
model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=0,
    epochs=20
)

print("\nCBOW Model Training Completed!")
print("Vocabulary Size:", len(model.wv))


# 6. Convert each tweet into a document vector
def get_document_vector(tokens):

    vectors = [
        model.wv[word]
        for word in tokens
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


document_vectors = np.array([
    get_document_vector(tokens)
    for tokens in tokenized_tweets
])

print("\nDocument Vectors Shape:", document_vectors.shape)


# 7. Take user input
user_input = input("\nEnter your interest or search query: ")

user_tokens = preprocess(user_input)

user_vector = get_document_vector(user_tokens)


# 8. Calculate cosine similarity

if np.linalg.norm(user_vector) == 0:

    print("No matching words found in vocabulary.")

else:

    similarities = cosine_similarity(
        user_vector.reshape(1, -1),
        document_vectors
    )[0]

    # 9. Get top 5 recommended tweets

    top_indices = np.argsort(similarities)[::-1][:5]

    print("\nTop 5 Recommended Tweets:\n")

    for rank, index in enumerate(top_indices, start=1):

        print("Recommendation", rank)
        print("Tweet:", tweets.iloc[index])
        print("Similarity Score:", similarities[index])
        print("-" * 60)


Dataset Shape: (10000, 6)

Sample Tokenized Tweets:
[['party', 'least', 'receive', 'say', 'or', 'single', 'prevent', 'prevent', 'husband', 'affect', 'may', 'himself', 'cup', 'style', 'evening', 'protect', 'effect', 'another', 'themselves', 'stage', 'perform', 'possible', 'try', 'tax', 'share', 'style', 'television', 'with', 'successful', 'much', 'sell', 'development', 'economy', 'effect'], ['hotel', 'still', 'congress', 'may', 'member', 'staff', 'media', 'draw', 'buy', 'fly', 'identify', 'on', 'another', 'turn', 'minute', 'would', 'local', 'subject', 'way', 'believe', 'which', 'question', 'some', 'message', 'own', 'all', 'imagine', 'join', 'agency', 'indicate'], ['nice', 'be', 'her', 'debate', 'industry', 'that', 'year', 'film', 'where', 'generation', 'push', 'discover', 'partner', 'level', 'nearly', 'money', 'store', 'style', 'may', 'enjoy', 'kid', 'discuss', 'blue', 'save', 'model', 'another', 'about', 'along', 'everybody', 'especially', 'dinner', 'character', 'yard']]

CBOW Model Tr


Enter your interest or search query:  technology



Top 5 Recommended Tweets:

Recommendation 1
Tweet: Though try away. Question we type young.
Since white modern financial news.
Consumer radio beyond argue. Always certain international computer strategy produce change.
Usually field rule call.
Reduce yourself fear. Space treat data team prove.
Similarity Score: 0.52114576
------------------------------------------------------------
Recommendation 2
Tweet: Land rest different product. Strong them enter white.
Behind management name top office beyond. Through expert technology test around.
Speech door carry next well tree data.
Rate though commercial opportunity. Affect vote although ask other.
Similarity Score: 0.50516456
------------------------------------------------------------
Recommendation 3
Tweet: Bed country total. East least exactly billion perform whom unit. Town around name technology experience laugh material issue.
Collection student understand plan beyond. Natural six believe general memory.
Similarity Score: 0.5037726
-